In [2]:
from transformers import pipeline
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoConfig
import numpy as np
from scipy.special import softmax


In [3]:
import google.protobuf
print("protobuf installed successfully")


protobuf installed successfully


In [4]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    truncation=True,
    max_length=512
)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
def preprocess(text):
    new_text = []
    for t in text.split(" "):
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    return " ".join(new_text)


In [6]:
MODEL = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)

model = AutoModelForSequenceClassification.from_pretrained(MODEL)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
text = "Good night 😊"
text = preprocess(text)

print("Processed text:", text)


Processed text: Good night 😊


In [8]:
encoded_input = tokenizer(text, return_tensors='pt')
encoded_input


{'input_ids': tensor([[    0, 18621, 17431,     6, 82803,     2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}

In [9]:
output = model(**encoded_input)


In [10]:
scores = output[0][0].detach().numpy()
scores = softmax(scores)

scores


array([0.03125937, 0.20148005, 0.7672607 ], dtype=float32)

In [11]:
ranking = np.argsort(scores)[::-1]

for i in range(scores.shape[0]):
    label = config.id2label[ranking[i]]
    score = scores[ranking[i]]
    print(f"{i+1}) {label} {np.round(float(score), 4)}")


1) positive 0.7673
2) neutral 0.2015
3) negative 0.0313


In [12]:
import pandas as pd

df = pd.read_csv("../datasets/custom_sentiment_dataset.csv")
df


,id,text,language_type,gold_label
0,1,Good night 😊,English,positive
1,2,I am feeling very sad today,English,negative
2,3,The weather is okay,English,neutral
3,4,শুভ রাত্রি 😊,Bengali,positive
4,5,আজ মনটা খুব খারাপ,Bengali,negative
5,6,আজ আবহাওয়া মোটামুটি,Bengali,neutral
6,7,Good না লাগতেছে আজ,CodeMixed,negative
7,8,আজ presentation ta ভালো হয়েছে 😊,CodeMixed,positive
8,9,Exam টা okay ছিল,CodeMixed,neutral
9,10,I love বাংলাদেশের মানুষ ❤️,CodeMixed,positive


# **BanglaBook Dataset Test with XMLr roBERTa**

In [14]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report

In [15]:
banglabook = pd.read_csv("../datasets/banglabook/csv/test.csv")

print(banglabook.head())


       id                                          Book_Name  \
0   72742   কম্পিউটার প্রোগ্রামিং ৩য় খণ্ড : ডেটা স্ট্রাকচ...   
1  100537   ম্যাসেজ (হার্ডকভার)  আধুনিক মননে দ্বীনের ছোঁয়...   
2   10441   মুক্তিযুদ্ধ নিয়ে স্মৃতিচারণমূলক ৫টি বই (হার্ড...   
3  126428                                      Bandhobi        
4   54626   A Clash of Kings (Book 2 Of A Song Of Ice And...   

              Writer_Name                                   Category  Rating  \
0   তামিম শাহরিয়ার সুবিন                       প্রোগ্রামিং বেসিক বই        1   
1   মিজানুর রহমান আজহারি                      ইসলামি আদর্শ  ও মতবাদ        5   
2        হাসান আজিজুল হক    মুক্তিযুদ্ধের ডায়েরি, চিঠি ও স্মৃতিচারণ        5   
3              Raba Khan                              English Story        1   
4    George R. R. Martin                    Novel: English Language        5   

                                              Review      Site sentiment  \
0  দারুন ‍একটা বই ,প্রথমে ভেবেছিলাম  Online থেকে ...  Roko

In [16]:
print(banglabook.columns)


Index(['id', 'Book_Name', 'Writer_Name', 'Category', 'Rating', 'Review',
       'Site', 'sentiment', 'label'],
      dtype='object')


In [18]:
banglabook = banglabook.rename(columns={
    "Review": "text",
    "label": "gold_label"
})


In [19]:
def normalize_label(label):
    if label == 0:
        return "negative"
    elif label == 1:
        return "neutral"
    elif label == 2:
        return "positive"
    else:
        return "neutral"

banglabook["gold_label"] = banglabook["gold_label"].apply(normalize_label)


In [20]:
banglabook = banglabook.dropna(subset=["text"])
banglabook["text"] = banglabook["text"].astype(str)


In [21]:
def predict_label(text):
    if not isinstance(text, str) or text.strip() == "":
        return "neutral"

    result = sentiment_task(text)[0]   # single dict
    return result["label"]


In [22]:
tqdm.pandas()

banglabook["predicted_label"] = banglabook["text"].progress_apply(predict_label)



100%|██████████| 31614/31614 [1:31:39<00:00,  5.75it/s]  


In [23]:
from sklearn.metrics import accuracy_score, classification_report

y_true = banglabook["gold_label"]
y_pred = banglabook["predicted_label"]

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))

Accuracy: 0.5514329094704877
              precision    recall  f1-score   support

    negative       0.22      0.41      0.29      1935
     neutral       0.05      0.41      0.09      1361
    positive       0.95      0.57      0.71     28318

    accuracy                           0.55     31614
   macro avg       0.41      0.46      0.36     31614
weighted avg       0.86      0.55      0.66     31614



# **Bengali_Sentiment Dataset Test with XMLr roBERTa**

In [31]:
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report

tqdm.pandas()

In [32]:
# Load files
with open("../datasets/bengali_sentiment/all_positive_8500.txt", "r", encoding="utf-8") as f:
    pos_texts = f.readlines()

with open("../datasets/bengali_sentiment/all_negative_3307.txt", "r", encoding="utf-8") as f:
    neg_texts = f.readlines()

# Clean text
pos_texts = [t.strip() for t in pos_texts if t.strip()]
neg_texts = [t.strip() for t in neg_texts if t.strip()]

# Create DataFrames
pos_df = pd.DataFrame({
    "text": pos_texts,
    "gold_label": "positive"
})

neg_df = pd.DataFrame({
    "text": neg_texts,
    "gold_label": "negative"
})

# Combine
bengali_sentiment = pd.concat([pos_df, neg_df], ignore_index=True)

print("Total samples:", len(bengali_sentiment))
bengali_sentiment.head()

Total samples: 11807


,text,gold_label
0,অসাধারণ নিশো বস্ আর অমি ভাইকেও।,positive
1,আমার দেখা বেস্ট নাটক,positive
2,"নাটক টা অনেক সুন্দর হয়েছে,,,,আফরান নিশো ভাইয়...",positive
3,সত্যি অসাধারণ একটি রিলেশন,positive
4,মজা পাইছি ভাষা গুলো কেমন লাগলো,positive


In [37]:
def predict_label(text):
    if not isinstance(text, str) or text.strip() == "":
        return "neutral"

    try:
        result = sentiment_model(
            text,
            truncation=True,
            max_length=512
        )[0]
        return result["label"]

    except Exception as e:
        print("Error on text:", text[:100])
        return "neutral"

In [38]:
from tqdm import tqdm
tqdm.pandas()

bengali_sentiment["predicted_label"] = (
    bengali_sentiment["text"].progress_apply(predict_label)
)

bengali_sentiment.head()

100%|██████████| 11807/11807 [32:47<00:00,  6.00it/s]    


,text,gold_label,predicted_label
0,অসাধারণ নিশো বস্ আর অমি ভাইকেও।,positive,positive
1,আমার দেখা বেস্ট নাটক,positive,neutral
2,"নাটক টা অনেক সুন্দর হয়েছে,,,,আফরান নিশো ভাইয়...",positive,positive
3,সত্যি অসাধারণ একটি রিলেশন,positive,positive
4,মজা পাইছি ভাষা গুলো কেমন লাগলো,positive,positive


In [40]:
from sklearn.metrics import accuracy_score, classification_report

labels = ["negative", "neutral", "positive"]

accuracy = accuracy_score(
    bengali_sentiment["gold_label"],
    bengali_sentiment["predicted_label"]
)

print("Accuracy:", accuracy)

print(
    classification_report(
        bengali_sentiment["gold_label"],
        bengali_sentiment["predicted_label"],
        labels=labels,
        digits=4,
        zero_division=0
    )
)

Accuracy: 0.7232997374438892
              precision    recall  f1-score   support

    negative     0.7044    0.5195    0.5980      3307
     neutral     0.0000    0.0000    0.0000         0
    positive     0.9543    0.8026    0.8719      8500

    accuracy                         0.7233     11807
   macro avg     0.5529    0.4407    0.4900     11807
weighted avg     0.8843    0.7233    0.7952     11807



# **Sentnob Dataset Test with XMLr roBERTa**

In [13]:
import pandas as pd

# Load CSVs
train_df = pd.read_csv("../datasets/sentnob/Train.csv")
val_df   = pd.read_csv("../datasets/sentnob/Val.csv")
test_df  = pd.read_csv("../datasets/sentnob/Test.csv")

# Rename columns
train_df = train_df.rename(columns={"Data": "text", "Label": "label"})
val_df   = val_df.rename(columns={"Data": "text", "Label": "label"})
test_df  = test_df.rename(columns={"Data": "text", "Label": "label"})

print(train_df.head())
print(train_df["label"].value_counts())

                                                text  label
0  মুগ্ধ হয়ে গেলাম মামু. আর তোমায় কি কমু. বলো তোম...      1
1  এই কুত্তার বাচ্চাদের জন্য দেশটা আজ এমন অবস্তায়...      2
2                          ভাই আপনার কথাই যাদু রয়েছে      1
3                        উওরটা আমার অনেক ভাল লেগেছে       1
4  আমার নিজের গাড়ী নিয়ে কি সাজেক যেতে পারবো না ?...      0
label
1    5133
2    4548
0    2894
Name: count, dtype: int64


In [14]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)
test_ds  = Dataset.from_pandas(test_df)

In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "cardiffnlp/twitter-xlm-roberta-base-sentiment"
)

In [16]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/12575 [00:00<?, ? examples/s]

Map:   0%|          | 0/1567 [00:00<?, ? examples/s]

Map:   0%|          | 0/1586 [00:00<?, ? examples/s]

In [17]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "cardiffnlp/twitter-xlm-roberta-base-sentiment",
    num_labels=3
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./sentnob_model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50
)

In [19]:
import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

In [21]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

trainer.train()

c:\Users\afrin\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,1.084764
100,1.043459
150,0.991291
200,0.991363
250,0.967438
300,0.894289
350,0.897076
400,0.956587
450,0.928240
500,0.932594


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import classification_report

predictions = trainer.predict(test_ds)
y_pred = predictions.predictions.argmax(axis=1)
y_true = test_df["label"]

print(classification_report(
    y_true,
    y_pred,
    target_names=["neutral", "positive", "negative"],
    digits=4
))